# Intro to DuckDB

Welcome! In this notebook you'll get hands-on with **DuckDB** — a powerful analytical database that runs entirely inside this Codespaces environment.

By the end of this notebook you'll be able to:
- Understand what DuckDB is and why it's useful for data warehousing
- Create tables and load data using SQL
- Explore your database structure with the information schema
- Run analytical queries and read data directly from CSV files

## What is DuckDB?

DuckDB is an **embedded analytical database** — similar to SQLite (no server to install or manage), but built specifically for **analytical workloads** like data warehousing.

| Feature | Traditional DB (e.g. PostgreSQL) | DuckDB |
|---|---|---|
| Installation | Server setup required | No setup — just Python |
| Best for | Transactions (INSERT, UPDATE) | Analytics (SELECT, GROUP BY, aggregations) |
| Speed on large datasets | Good | Very fast (columnar storage) |
| Where it runs | On a server | Right here in your notebook |

DuckDB speaks **standard SQL** — everything you already know works here.

## Why use DuckDB for this training?

1. **No infrastructure required** — runs inside Codespaces, no database servers, no credentials, no IT tickets.
2. **Same SQL patterns as production** — CTEs, window functions, GROUP BY, and JOIN behave identically to BigQuery, Snowflake, and Redshift.
3. **Software engineering principles for data** — we can create schemas, enforce constraints, and build layered data models (just like a dbt project) all in one place.
4. **Fast feedback loop** — changes are instant, mistakes are easy to reset, and you can experiment freely.

> The goal isn't to make you a DuckDB expert — it's to teach **data warehousing patterns** using a tool that gets out of the way.

## A note on Python

You don't need to know Python to use these notebooks. Python is just the **wrapper** that sends your SQL to DuckDB and displays the results.

The only pattern you need to remember:

```python
con.execute('''
    YOUR SQL HERE
''').df()
```

- `con.execute(...)` — sends your SQL to DuckDB
- `.df()` — returns the result as a table you can read

That's it. Every code cell in these notebooks follows this pattern.

## 1. Connect to DuckDB

In [ ]:
import duckdb

# In-memory database — great for learning, no files created
# To persist data to disk, pass a filename: duckdb.connect('mydb.duckdb')
con = duckdb.connect()

print('Connected! DuckDB version:', duckdb.__version__)

## 2. Creating Tables

DuckDB uses standard SQL DDL — the same `CREATE TABLE` syntax you already know.

Let's create two tables to work with: `products` and `orders`.

In [ ]:
con.execute('''
    CREATE TABLE products (
        product_id   INTEGER,
        product_name VARCHAR,
        category     VARCHAR,
        unit_price   DECIMAL(10, 2)
    )
''')

con.execute('''
    INSERT INTO products VALUES
        (1, 'Espresso Machine', 'Appliances',  249.99),
        (2, 'Coffee Grinder',   'Appliances',   89.99),
        (3, 'French Press',     'Kitchenware',  29.99),
        (4, 'Coffee Beans 1kg', 'Consumables',  24.99),
        (5, 'Milk Frother',     'Appliances',   39.99)
''')

print('products table created.')

In [ ]:
con.execute('''
    CREATE TABLE orders (
        order_id   INTEGER,
        product_id INTEGER,
        order_date DATE,
        quantity   INTEGER,
        region     VARCHAR
    )
''')

con.execute('''
    INSERT INTO orders VALUES
        (101, 1, '2024-01-10', 1, 'North'),
        (102, 4, '2024-01-11', 3, 'South'),
        (103, 3, '2024-01-12', 2, 'East'),
        (104, 2, '2024-01-14', 1, 'West'),
        (105, 5, '2024-01-15', 2, 'North'),
        (106, 4, '2024-02-02', 5, 'South'),
        (107, 1, '2024-02-05', 1, 'East'),
        (108, 3, '2024-02-08', 4, 'West'),
        (109, 2, '2024-02-10', 2, 'North'),
        (110, 5, '2024-02-14', 1, 'South')
''')

print('orders table created.')

In [ ]:
# Preview the products table
con.execute('SELECT * FROM products').df()

In [ ]:
# Preview the orders table
con.execute('SELECT * FROM orders').df()

## 3. Exploring Your Database Structure

Once you've created tables, you'll often need to see what exists and how it's structured.
DuckDB gives you several ways to inspect your database.

In [ ]:
# See all tables in the current database
con.execute('SHOW TABLES').df()

In [ ]:
# information_schema.tables gives richer detail (schema, table type, etc.)
con.execute('SELECT * FROM information_schema.tables').df()

In [ ]:
# information_schema.columns shows every column, its type, and position
con.execute('''
    SELECT
        table_name,
        column_name,
        data_type,
        ordinal_position
    FROM information_schema.columns
    ORDER BY table_name, ordinal_position
''').df()

## 4. Running Analytical Queries

DuckDB is built for this — fast aggregations, JOINs, and window functions on large datasets.

In [ ]:
# Total revenue and number of orders by region
con.execute('''
    SELECT
        o.region,
        COUNT(o.order_id)              AS num_orders,
        SUM(o.quantity * p.unit_price) AS total_revenue
    FROM orders o
    JOIN products p ON o.product_id = p.product_id
    GROUP BY o.region
    ORDER BY total_revenue DESC
''').df()

In [ ]:
# Window function: running total of revenue over time
con.execute('''
    SELECT
        o.order_date,
        o.order_id,
        p.product_name,
        o.quantity * p.unit_price                        AS order_revenue,
        SUM(o.quantity * p.unit_price) OVER (
            ORDER BY o.order_date
        )                                                AS running_total
    FROM orders o
    JOIN products p ON o.product_id = p.product_id
    ORDER BY o.order_date
''').df()

## 5. Reading Directly from CSV Files

One of DuckDB's most useful features: you can query CSV files directly, without loading them into a table first.
This is how we'll load source data in the exercises.

In [ ]:
# Query a CSV file directly — DuckDB infers the schema automatically
con.execute('''
    SELECT *
    FROM read_csv_auto('../data/raw_customers.csv')
    LIMIT 5
''').df()

In [ ]:
# You can also CREATE a table directly from a CSV
con.execute('''
    CREATE OR REPLACE TABLE raw_customers AS
    SELECT * FROM read_csv_auto('../data/raw_customers.csv')
''')

con.execute('SELECT COUNT(*) AS total_customers FROM raw_customers').df()

---
## Give it a go!

Remember the `products` and `orders` tables from earlier? 

In [ ]:
con.execute('''SELECT * FROM products''').df()

In [ ]:
con.execute('''SELECT * FROM orders''').df()

Use these to answer the questions below.
The Python wrapper is already written — just fill in the SQL.

> **Tip:** If you get stuck, look back at the query examples above. The patterns are the same.

**Challenge 1:** How many orders were placed for each product? Show the product name and order count, sorted by most popular first.

In [ ]:
con.execute('''
    SELECT
        p.product_name,
        COUNT(o.order_id) AS num_orders
    FROM orders o
    JOIN products p ON o.product_id = p.product_id
    GROUP BY p.product_name
    ORDER BY num_orders DESC
''').df()

**Challenge 2:** What is the total revenue per product category? Show the category and total revenue, sorted highest first.

In [ ]:
con.execute('''
    SELECT
        p.category,
        SUM(o.quantity * p.unit_price) AS total_revenue
    FROM orders o
    JOIN products p ON o.product_id = p.product_id
    GROUP BY p.category
    ORDER BY total_revenue DESC
''').df()

**Challenge 3:** Which single order had the highest revenue? Return the order ID, product name, and revenue for that order only.

In [ ]:
con.execute('''
    SELECT
        o.order_id,
        p.product_name,
        o.quantity * p.unit_price AS revenue
    FROM orders o
    JOIN products p ON o.product_id = p.product_id
    ORDER BY revenue DESC
    LIMIT 1
''').df()

**Challenge 4 (stretch):** For each order, show the order revenue and its rank compared to all other orders (1 = highest revenue). Use a window function.

In [ ]:
con.execute('''
    SELECT
        o.order_id,
        p.product_name,
        o.quantity * p.unit_price                              AS revenue,
        RANK() OVER (ORDER BY o.quantity * p.unit_price DESC) AS revenue_rank
    FROM orders o
    JOIN products p ON o.product_id = p.product_id
    ORDER BY revenue_rank
''').df()

## Summary

| What you used | What it does |
|---|---|
| `duckdb.connect()` | Opens an in-memory database |
| `con.execute('...').df()` | Runs SQL, returns a table |
| `CREATE TABLE` | Creates a table with defined columns |
| `SHOW TABLES` | Lists all tables in the database |
| `information_schema.tables` | Table metadata (schema, type) |
| `information_schema.columns` | Column metadata (name, type, position) |
| `read_csv_auto('file.csv')` | Queries a CSV without loading it into a table |
| `CREATE TABLE ... AS SELECT` | Creates a table from a query result |

---